# 🇧🇷 Lloavece-Coder: Treinamento na Nuvem (Colab Pro + Drive 5TB)

Este notebook executa o pipeline industrial de treinamento e compressão da **Lloavece-Coder** utilizando aceleração por GPU (A100 / L4) e salvamento automático atômico no Google Drive.

### 1. Conectar ao Google Drive de 5TB (Persistência Total)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/lloavece_checkpoints
print("[*] Google Drive conectado com sucesso!")

### 2. Verificar o Monstro de GPU (A100 / L4)

In [ ]:
!nvidia-smi

### 3. Clonar ou Acessar o Repositório Lloavece

In [ ]:
import os
if not os.path.exists('Lloavece'):
    !git clone https://github.com/alekkzsx/Lloavece.git
%cd Lloavece
!pip install sentencepiece datasets tqdm numpy crcmod -q
print("[*] Ambiente Lloavece pronto para batalha!")

### 4. Coleta de Código e Treinamento do Tokenizer de Programação

In [ ]:
# Coleta datasets de alta fidelidade de código e regras de negócio BR
!python -m src.data.collectors.download_code --samples 50000

# Treina o Tokenizer Byte-Fallback BPE com preservação de indentação e FIM
!python -m src.tokenizer.code_tokenizer --samples 100000

# Converte tudo em binário puro mapeado em memória (uint16 Memmap)
!python -m src.data.pipeline --fim_rate 0.5

### 5. Iniciar Pré-Treinamento (bfloat16 + Backup no Drive)

In [ ]:
!python -m src.training.train \
    --config 0.5B \
    --precision bf16 \
    --batch_size 8 \
    --grad_accum_steps 16 \
    --max_iters 10000 \
    --drive_dir /content/drive/MyDrive/lloavece_checkpoints

### 6. Fine-Tuning de Instruções Técnicas em Português (SFT)

In [ ]:
!python -m src.training.sft \
    --checkpoint checkpoints/best_model.pt \
    --drive_dir /content/drive/MyDrive/lloavece_checkpoints

### 7. Comprimir o Modelo com o Motor K-Bit Sparse da Lloavece

In [ ]:
!python -m src.export.kbit_compressor

### 8. Teste de Geração de Código em Português

In [ ]:
!python -m src.generate \
    --checkpoint checkpoints/best_model.pt \
    --prompt "def processar_pagamento_pix(chave: str, valor: float):"